# Hito 2 - Notebook 06: Preparacion de Datos - Frente Logistica
## Fase 3 de CRISP-DM

Orden: **Carga -> Limpieza -> Transformacion (mapeo ALDIMI + series temporales + targets de demanda t+7/t+14) -> Reduccion**.

> **Decision de diseno (Fase 1/3):** el objetivo de regresion es la **demanda (consumo) acumulada futura** a 7 y 14 dias, no el nivel absoluto de stock. El EDA mostro que el stock a ese horizonte no tiene autocorrelacion (los cambios diarios son ruido), mientras que el consumo es altamente predecible (autocorrelacion ~0.90; consumo de la semana previa ~ consumo de la semana siguiente, corr ~0.96). El **stock proyectado y las alertas se derivan** de la demanda predicha (`Stock_Proyectado = Stock_Actual - Demanda_Predicha`).

In [ ]:
import sys
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'src' / 'aldimi_common.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import aldimi_common as ac
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)
print('Raiz del proyecto:', ROOT)

## 1.4.1 Carga de Datos

In [ ]:
stock = pd.read_csv(ac.DATA_RAW / ac.STOCK_RAW_FILE)
print('Datos crudos:', stock.shape)
stock.head(3)

## 1.4.2 Limpieza de Datos (Data Cleaning)

In [ ]:
print('Nulos:', int(stock.isna().sum().sum()), '| Duplicados:', int(stock.duplicated().sum()))
# Stockout_Flag es constante (todos 0): no aporta informacion -> se descarta
if stock['Stockout_Flag'].nunique() == 1:
    stock = stock.drop(columns=['Stockout_Flag'])
    print('Columna Stockout_Flag eliminada (varianza nula).')
print('Tras limpieza:', stock.shape)

> **Conclusion limpieza:** sin nulos ni duplicados; se elimina `Stockout_Flag` por varianza nula (no aporta al modelo).

## 1.4.3 Transformacion de Datos

### Mapeo al contexto ALDIMI y consolidacion por insumo/dia

In [ ]:
stock = ac.map_insumos(stock)
stock = ac.aggregate_daily_insumo(stock)
print('Serie consolidada:', stock.shape)
stock[['Fecha', 'ID_Insumo', 'Insumo', 'Categoria_Insumo', 'Stock_Actual', 'Consumo_Diario']].head()

### Variables contextuales del albergue (integracion entre frentes)

In [ ]:
stock = ac.add_occupancy_context(stock)
stock[['Fecha', 'Ocupacion_Total', 'Ocupacion_Albergue', 'Pacientes_Alto_Riesgo']].drop_duplicates('Fecha').head()

> `Ocupacion_Total` modela la transicion **ALDIMI 2.0 (50 -> 100 familias)** y conecta el frente clinico con el logistico (mas pacientes -> mayor consumo).

### Feature engineering de series temporales

Se agregan **sumas moviles del consumo pasado** (`Consumo_Prev_7d`, `Consumo_Prev_14d`), su **volatilidad** (`Consumo_Std_7d`) y **rezagos** (`Consumo_Lag_1`, `Consumo_Lag_7`), que son los predictores mas fuertes de la demanda futura.

In [ ]:
stock = ac.add_stock_features(stock)
nuevas = ['Consumo_7d', 'Consumo_14d', 'Consumo_Prev_7d', 'Consumo_Prev_14d',
          'Consumo_Std_7d', 'Consumo_Lag_1', 'Consumo_Lag_7', 'Stock_Lag_1',
          'Ratio_Stock', 'Cobertura_Dias', 'Mes', 'Dia_Semana', 'Alerta']
stock[nuevas].head()

### Construccion de los objetivos de regresion: demanda futura (t+7 y t+14)

In [ ]:
stock = ac.build_demand_targets(stock)
print('Objetivos creados:', ac.DEMAND_TARGET_7, ac.DEMAND_TARGET_14)
stock[['Fecha', 'ID_Insumo', 'Consumo_Diario', 'Consumo_Prev_7d',
       ac.DEMAND_TARGET_7, ac.DEMAND_TARGET_14]].head(10)

> Los targets son la **demanda (consumo) acumulada futura** por insumo: la suma del consumo diario en los proximos 7 y 14 dias. Las ultimas fechas de cada insumo quedan sin target (NaN) y se excluyen en el modelado. La decision operativa de compra se deriva luego: `Stock_Proyectado = Stock_Actual - Demanda_Predicha`, y si cae bajo el punto de reorden se dispara la alerta.

### Escalado (demostracion)

In [ ]:
from sklearn.preprocessing import StandardScaler
feats = ac.stock_feature_columns(stock)
demo = pd.DataFrame(StandardScaler().fit_transform(stock[feats].fillna(0)), columns=feats)
print('Features de regresion:', feats)
demo.describe().T[['mean', 'std']].round(3).head()

> **Conclusion transformacion:** se generan promedios moviles, rezagos, ratios y componentes temporales. El escalado se aplicara dentro del Pipeline de modelado. Los modelos de arboles (RF/XGBoost) son robustos a la escala.

## 1.4.4 Reduccion de Datos

In [ ]:
print('Se descartan del modelo (identificadores / target / alertas):', ac.STOCK_EXCLUDE_COLS)
print(f'Features finales de regresion: {len(feats)}')
feats

> **Conclusion reduccion:** se conservan variables operativas interpretables (consumo, cobertura, lead time, ocupacion) por su valor de negocio.

## Guardado del dataset preparado

In [ ]:
ac.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
salida = ac.DATA_PROCESSED / ac.STOCK_PROCESSED_FILE
stock.to_csv(salida, index=False)
print('Guardado:', salida, stock.shape)

> **Salida:** `data/processed/Dataset_ALDIMI_Logistica_Preparado.csv`.